In [1]:
import pandas as pd

url = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"

df = pd.read_csv(url, compression="zip")

print("Dataset Shape:", df.shape)
df.head()


Dataset Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [2]:
print(df.isnull().sum())

print(df['Machine failure'].value_counts())

df = df.drop(columns=[
    'UDI', 'Product ID',
    'TWF', 'HDF', 'PWF', 'OSF', 'RNF'
])

df = pd.get_dummies(df, columns=['Type'], dtype=int)

print("Updated Shape:", df.shape)
df.head()


UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64
Machine failure
0    9661
1     339
Name: count, dtype: int64
Updated Shape: (10000, 9)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Type_H,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,1
1,298.2,308.7,1408,46.3,3,0,0,1,0
2,298.1,308.5,1498,49.4,5,0,0,1,0
3,298.2,308.6,1433,39.5,7,0,0,1,0
4,298.2,308.7,1408,40.0,9,0,0,1,0


In [3]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


X = df.drop('Machine failure', axis=1)
y = df['Machine failure']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.96      0.40      0.56        68

    accuracy                           0.98      2000
   macro avg       0.97      0.70      0.78      2000
weighted avg       0.98      0.98      0.97      2000

Confusion Matrix:
[[1931    1]
 [  41   27]]


In [4]:

from sklearn.metrics import classification_report, confusion_matrix

# Predict failure probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Lower the threshold to detect more failures
y_pred_new = (y_prob >= 0.25).astype(int)

print(classification_report(y_test, y_pred_new))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_new))


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.68      0.74      0.70        68

    accuracy                           0.98      2000
   macro avg       0.83      0.86      0.85      2000
weighted avg       0.98      0.98      0.98      2000

Confusion Matrix:
[[1908   24]
 [  18   50]]


In [5]:

def assess_machine(data):
    probability = model.predict_proba(data)[0][1]

    if probability >= 0.60:
        risk = "High Risk"
    elif probability >= 0.25:
        risk = "Medium Risk"
    else:
        risk = "Low Risk"

    recommendations = []

    if data['Tool wear [min]'].iloc[0] > 200:
        recommendations.append("Inspect or replace worn tools.")

    if data['Torque [Nm]'].iloc[0] > 55:
        recommendations.append("Check mechanical load and torque.")

    if data['Rotational speed [rpm]'].iloc[0] < 1300:
        recommendations.append("Inspect motor speed and drive system.")

    if data['Process temperature [K]'].iloc[0] > 315:
        recommendations.append("Inspect cooling and temperature control.")

    if risk == "High Risk":
        recommendations.append("Prioritize maintenance inspection.")

    if not recommendations:
        recommendations.append("Continue routine monitoring.")

    return probability, risk, recommendations


In [6]:

sample = X_test.iloc[[0]]

probability, risk, recommendations = assess_machine(sample)

print("Failure Probability:", round(probability * 100, 2), "%")
print("Risk Level:", risk)

print("\nRecommendations:")
for recommendation in recommendations:
    print("-", recommendation)


Failure Probability: 17.0 %
Risk Level: Low Risk

Recommendations:
- Check mechanical load and torque.


In [7]:

import joblib

# Save trained model
joblib.dump(model, "machineguard_model.pkl")

# Save feature names in the correct order
joblib.dump(X.columns.tolist(), "feature_names.pkl")

print("Model and feature names saved successfully!")


Model and feature names saved successfully!


In [8]:

from google.colab import files

files.download("machineguard_model.pkl")
files.download("feature_names.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>